In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import tensorflow as tf

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

2025-11-21 17:17:17.458379: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763745437.653315      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763745437.715483      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/kaggle/input/tmdb-movie-metadata/tmdb_5000_movies.csv
/kaggle/input/tmdb-movie-metadata/tmdb_5000_credits.csv
/kaggle/input/movielens-100k/ml-latest-small/movies.csv
/kaggle/input/movielens-100k/ml-latest-small/ratings.csv
/kaggle/input/movielens-100k/ml-latest-small/README.txt
/kaggle/input/movielens-100k/ml-latest-small/tags.csv
/kaggle/input/movielens-100k/ml-latest-small/links.csv


In [2]:
movies_df = pd.read_csv("/kaggle/input/tmdb-movie-metadata/tmdb_5000_movies.csv")
#movies_df2 = pd.read_csv("/kaggle/input/top-500-600-movies-of-each-year-from-1960-to-2024/final_dataset.csv")
credits_df = pd.read_csv("/kaggle/input/tmdb-movie-metadata/tmdb_5000_credits.csv")
ratings_df = pd.read_csv("/kaggle/input/movielens-100k/ml-latest-small/ratings.csv")
links_df = pd.read_csv("/kaggle/input/movielens-100k/ml-latest-small/links.csv")

credits_df.columns

Index(['movie_id', 'title', 'cast', 'crew'], dtype='object')

In [3]:
# Load a pre-trained SBERT model.
# 'all-MiniLM-L6-v2' is fast and provides high-quality semantic embeddings.
print("Loading Sentence-Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully.")

Loading Sentence-Transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.


In [4]:
import pandas as pd
import numpy as np
import ast
from sklearn.preprocessing import MinMaxScaler

print("--- PARTIE 1 : PRÉPARATION ET FUSION ---")

# 1. Sécurisation des IDs pour la fusion
movies_df['id'] = pd.to_numeric(movies_df['id'], errors='coerce').astype('Int64')
credits_df['movie_id'] = pd.to_numeric(credits_df['movie_id'], errors='coerce').astype('Int64')

# 2. Fusion (Merge) : On ajoute Cast & Crew à Movies
# On utilise un left join pour ne pas perdre de films s'ils n'ont pas de crédits
movies_df = movies_df.merge(credits_df[['movie_id', 'cast', 'crew']], left_on='id', right_on='movie_id', how='left')

# 3. Fonctions de Parsing (Extraction depuis le format string/JSON)
def parse_list_col(x):
    try:
        if isinstance(x, str):
            # Transforme "[{'id': 1, 'name': 'Action'}]" en liste Python
            L = ast.literal_eval(x)
            # Renvoie la liste des noms ['Action', 'Adventure']
            return [i['name'] for i in L]
        return []
    except:
        return []

def get_director(x):
    try:
        if isinstance(x, str): x = ast.literal_eval(x)
        for i in x:
            if i['job'] == 'Director':
                return i['name']
        return np.nan
    except:
        return np.nan

def get_top_cast(x):
    try:
        if isinstance(x, str): x = ast.literal_eval(x)
        # On garde les 4 premiers acteurs
        return [i['name'] for i in x][:4]
    except:
        return []

# 4. Application des extractions
print("Extraction des métadonnées complexes...")
movies_df['genres_list'] = movies_df['genres'].apply(parse_list_col)
movies_df['keywords_list'] = movies_df['keywords'].apply(parse_list_col)
movies_df['countries_list'] = movies_df['production_countries'].apply(parse_list_col)
movies_df['director'] = movies_df['crew'].apply(get_director)
movies_df['cast_list'] = movies_df['cast'].apply(get_top_cast)

# Extraction de l'année
movies_df['release_date'] = pd.to_datetime(movies_df['release_date'], errors='coerce')
movies_df['year'] = movies_df['release_date'].dt.year.fillna(0)

# 5. Nettoyage des Espaces (Sanitization)
# Crucial pour CountVectorizer : "Science Fiction" -> "ScienceFiction"
# Sinon "Science" matcherait avec "Political Science"
def sanitize(x):
    if isinstance(x, list):
        return [str(i).replace(" ", "").lower() for i in x]
    else:
        if isinstance(x, str):
            return str(x).replace(" ", "").lower()
        return ''

movies_df['director_sanitized'] = movies_df['director'].apply(sanitize)
movies_df['cast_sanitized'] = movies_df['cast_list'].apply(sanitize)
movies_df['genres_sanitized'] = movies_df['genres_list'].apply(sanitize)
movies_df['keywords_sanitized'] = movies_df['keywords_list'].apply(sanitize)
movies_df['countries_sanitized'] = movies_df['countries_list'].apply(sanitize)

print("Données nettoyées et prêtes.")

--- PARTIE 1 : PRÉPARATION ET FUSION ---
Extraction des métadonnées complexes...
Données nettoyées et prêtes.


In [5]:
print("\n--- PARTIE 2 : CRÉATION DES SOUPES ---")

# 1. Soupe Sémantique (Pour SBERT) -> Contenu riche
movies_df['soup_overview'] = (
    movies_df['title'].fillna('') + ". " + 
    movies_df['tagline'].fillna('') + ". " + 
    movies_df['overview'].fillna('')
)

# 2. Soupe Mots-Clés (Pour CountVectorizer) -> Sujets précis
movies_df['soup_keywords'] = movies_df['keywords_sanitized'].apply(lambda x: ' '.join(x))

# 3. Soupe Crédits (Pour CountVectorizer) -> Style artistique
# On répète le réalisateur 3 fois pour lui donner plus de poids que les acteurs
def create_credits_soup(x):
    director = (x['director_sanitized'] + ' ') * 3
    cast = ' '.join(x['cast_sanitized'])
    return director + ' ' + cast

movies_df['soup_credits'] = movies_df.apply(create_credits_soup, axis=1)

# 4. Soupe Genre (Pour CountVectorizer) -> Catégorie
movies_df['soup_genres'] = movies_df['genres_sanitized'].apply(lambda x: ' '.join(x))

# 5. Soupe Origine (Pour CountVectorizer) -> Langue & Pays
def create_origin_soup(x):
    lang = str(x['original_language'])
    countries = ' '.join(x['countries_sanitized'])
    return lang + ' ' + countries

movies_df['soup_origin'] = movies_df.apply(create_origin_soup, axis=1)

print("Soupes (Textes composites) générées.")


--- PARTIE 2 : CRÉATION DES SOUPES ---
Soupes (Textes composites) générées.


In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import os

print("\n--- PARTIE 3 : CALCUL DES MATRICES ---")

# --- MOTEUR 1 : SBERT (Sémantique / Synopsis) ---
print("1. Calcul SBERT (Overview)...")
EMBEDDINGS_FILE = 'sbert_embeddings.npy'

# IMPORTANT : On s'assure d'abord que l'index est parfaitement aligné
# C'est la ligne CRUCIALE qui manquait pour garantir la synchro
movies_df = movies_df.reset_index(drop=True)

# Préparation du texte
movies_df['soup_overview'] = movies_df['title'] + ". " + movies_df['overview'].fillna('')

recalc = True # Par défaut, on recalcule

if os.path.exists(EMBEDDINGS_FILE):
    loaded_data = np.load(EMBEDDINGS_FILE)
    
    # --- CORRECTIF --- 
    # On compare la taille du fichier chargé avec la taille du DataFrame actuel
    if len(loaded_data) == len(movies_df):
        sbert_embeddings = loaded_data
        print(f"   Embeddings chargés et alignés ({len(loaded_data)} films).")
        recalc = False
    else:
        print(f"⚠️ TAILLE INCORRECTE : Fichier ({len(loaded_data)}) vs DataFrame ({len(movies_df)})")
        print("   >>> Ignorer le fichier et recalculer pour aligner.")

if recalc:
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    sbert_embeddings = sbert_model.encode(movies_df['soup_overview'].tolist(), show_progress_bar=True)
    np.save(EMBEDDINGS_FILE, sbert_embeddings)

sim_matrix_sbert = cosine_similarity(sbert_embeddings)


# --- MOTEUR 2 : KEYWORDS (Sujets) ---
print("2. Calcul Keywords...")
count_vec_keys = CountVectorizer(stop_words='english', min_df=2)
# min_df=2 enlève les keywords qui n'apparaissent qu'une seule fois (bruit)
matrix_keys = count_vec_keys.fit_transform(movies_df['soup_keywords'])
sim_matrix_keywords = cosine_similarity(matrix_keys, matrix_keys)


# --- MOTEUR 3 : CREDITS (Réalisateur + Cast) ---
print("3. Calcul Credits...")
count_vec_credits = CountVectorizer(stop_words='english')
matrix_credits = count_vec_credits.fit_transform(movies_df['soup_credits'])
sim_matrix_credits = cosine_similarity(matrix_credits, matrix_credits)


# --- MOTEUR 4 : GENRE ---
print("4. Calcul Genres...")
count_vec_genre = CountVectorizer(stop_words='english')
matrix_genre = count_vec_genre.fit_transform(movies_df['soup_genres'])
sim_matrix_genre = cosine_similarity(matrix_genre, matrix_genre)


# --- MOTEUR 5 : ORIGINE (Langue + Pays) ---
print("5. Calcul Origine...")
count_vec_origin = CountVectorizer()
matrix_origin = count_vec_origin.fit_transform(movies_df['soup_origin'])
sim_matrix_origin = cosine_similarity(matrix_origin, matrix_origin)


# --- MOTEUR 6 : CONTEXTE (Popularité + Date) ---
print("6. Calcul Contexte (Pop & Date)...")
# Distance Gaussienne : exp(-|diff| / sigma)
# Popularité
pop_vals = pd.to_numeric(movies_df['popularity'], errors='coerce').fillna(0).values.reshape(-1, 1)
pop_diff = np.abs(pop_vals - pop_vals.T)
sim_matrix_pop = np.exp(-pop_diff / 20) # Sigma=20 : tolérance moyenne

# Date (Année)
year_vals = movies_df['year'].values.reshape(-1, 1)
year_diff = np.abs(year_vals - year_vals.T)
sim_matrix_year = np.exp(-year_diff / 10) # Sigma=10 ans

# Moyenne des deux
sim_matrix_context = (sim_matrix_pop + sim_matrix_year) / 2

print("Toutes les matrices sont prêtes.")


--- PARTIE 3 : CALCUL DES MATRICES ---
1. Calcul SBERT (Overview)...


Batches:   0%|          | 0/151 [00:00<?, ?it/s]

2. Calcul Keywords...
3. Calcul Credits...
4. Calcul Genres...
5. Calcul Origine...
6. Calcul Contexte (Pop & Date)...
Toutes les matrices sont prêtes.


In [7]:
print("\n--- PARTIE 4 : MOTEUR DE RECOMMANDATION ---")

# Indexation pour recherche rapide
indices = pd.Series(movies_df.index, index=movies_df['title']).drop_duplicates()

def get_recommendations(title, weights, N=5):
    """
    Génère des recommandations basées sur une somme pondérée de similarités.
    
    weights : dict avec les clés 'sbert', 'keywords', 'credits', 'genre', 'origin', 'context'
              La somme des poids n'a pas besoin de faire 1 (on classe par score relatif).
    """
    # 1. Récupération de l'index
    try:
        idx = indices[title]
        # Gestion cas films homonymes (on prend le plus récent/populaire si trié)
        if isinstance(idx, pd.Series): idx = idx.iloc[0]
    except KeyError:
        print(f"Erreur : Film '{title}' introuvable.")
        return

    # 2. Calcul du Score Final (Weighted Average)
    # C'est ici que la magie opère : on mixe les matrices selon vos désirs
    final_scores = (
        weights.get('sbert', 0)    * sim_matrix_sbert[idx] +
        weights.get('keywords', 0) * sim_matrix_keywords[idx] +
        weights.get('credits', 0)  * sim_matrix_credits[idx] +
        weights.get('genre', 0)    * sim_matrix_genre[idx] +
        weights.get('origin', 0)   * sim_matrix_origin[idx] +
        weights.get('context', 0)  * sim_matrix_context[idx]
    )

    # 3. Tri et Extraction
    scores = list(enumerate(final_scores))
    # Tri décroissant
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    # On prend les N suivants (exclure le premier qui est le film lui-même)
    top_indices = [i[0] for i in scores[1:N+1]]
    
    # 4. Affichage Riche
    print(f"\n=== RECOMMANDATIONS POUR : {title} ===")
    print(f"Configuration : {weights}")
    print("-" * 60)
    
    rec_movies = movies_df.iloc[top_indices]
    
    for i, row in rec_movies.iterrows():
        # Récupération du score brut
        score = final_scores[i]
        
        print(f"[{score:.3f}] {row['title']} ({int(row['year'])})")
        print(f"        Director: {row['director']}")
        print(f"        Genre: {row['genres_list'][:3]}")
        print(f"        Keywords: {row['keywords_list'][:4]}")
        print(f"        Origin: {row['original_language']} | Pop: {float(row['popularity']):.1f}")
        print("-" * 30)


--- PARTIE 4 : MOTEUR DE RECOMMANDATION ---


In [8]:
# SCÉNARIO : Votre demande spécifique
# - Keywords très importants (Sujet précis)
# - Meta important (Genre, Langue Coréenne, etc.)
# - SBERT moyen (Contenu global)
# - Context faible (Date/Pop juste pour filtrer les incohérences)

# Remplacer la définition de my_weights par ceci :
my_weights = {
    'sbert': 0.15,    
    'keywords': 0.40, 
    'genre': 0.10,    # Au lieu de 'meta'
    'origin': 0.05,   # Au lieu de 'meta'
    'credits': 0.10,  # Au lieu de 'meta' (Total meta = 0.30)
    'context': 0.15   
}

# Test 1 : Thriller Coréen
# Oldboy (2003) est un excellent test.
print("\n--- TEST 1 : Thriller Coréen ---")
get_recommendations('Star Trek', my_weights)

# Test 2 : Animation / Pixar
print("\n--- TEST 2 : Animation ---")
get_recommendations('Prisoners', my_weights)

# Test 3 : Science Fiction complexe
print("\n--- TEST 3 : Sci-Fi ---")
get_recommendations('Good Will Hunting', my_weights)


--- TEST 1 : Thriller Coréen ---

=== RECOMMANDATIONS POUR : Star Trek ===
Configuration : {'sbert': 0.15, 'keywords': 0.4, 'genre': 0.1, 'origin': 0.05, 'credits': 0.1, 'context': 0.15}
------------------------------------------------------------
[0.527] Star Trek Into Darkness (2013)
        Director: J.J. Abrams
        Genre: ['Action', 'Adventure', 'Science Fiction']
        Keywords: ['spacecraft', 'friendship', 'sequel', 'futuristic']
        Origin: en | Pop: 78.3
------------------------------
[0.373] Star Trek IV: The Voyage Home (1986)
        Director: Leonard Nimoy
        Genre: ['Science Fiction', 'Adventure']
        Keywords: ['saving the world', 'san francisco', 'uss enterprise-a', 'time travel']
        Origin: en | Pop: 22.3
------------------------------
[0.357] Star Trek Beyond (2016)
        Director: Justin Lin
        Genre: ['Action', 'Adventure', 'Science Fiction']
        Keywords: ['sequel', 'stranded', 'hatred', 'space opera']
        Origin: en | Pop: 65

In [9]:
# --- CONFIGURATIONS DE POIDS ---

# Cas 1 : "Je veux exactement le même sujet" (Priorité Keywords)
w_topic = {
    'sbert': 0.2, 'keywords': 0.5, 'genre': 0.1, 
    'credits': 0.1, 'origin': 0.0, 'context': 0.1
}

# Cas 2 : "Je veux le même réalisateur / casting" (Priorité Credits)
w_director = {
    'sbert': 0.1, 'keywords': 0.1, 'genre': 0.1, 
    'credits': 0.6, 'origin': 0.0, 'context': 0.1
}

# Cas 3 : "Je veux un film Coréen du même genre" (Priorité Origine + Genre)
w_korean = {
    'sbert': 0.1, 'keywords': 0.1, 'genre': 0.3, 
    'credits': 0.1, 'origin': 0.4, 'context': 0.0
}

# TESTS
get_recommendations('Interstellar', w_director, N=4) # Devrait donner Inception/Nolan
get_recommendations('Oldboy', w_korean, N=4)         # Devrait donner des thrillers coréens
get_recommendations('Toy Story', w_topic, N=4)       # Devrait donner Toy Story 2/3


=== RECOMMANDATIONS POUR : Interstellar ===
Configuration : {'sbert': 0.1, 'keywords': 0.1, 'genre': 0.1, 'credits': 0.6, 'origin': 0.0, 'context': 0.1}
------------------------------------------------------------
[0.587] The Dark Knight Rises (2012)
        Director: Christopher Nolan
        Genre: ['Action', 'Crime', 'Drama']
        Keywords: ['dc comics', 'crime fighter', 'terrorist', 'secret identity']
        Origin: en | Pop: 112.3
------------------------------
[0.543] The Prestige (2006)
        Director: Christopher Nolan
        Genre: ['Drama', 'Mystery', 'Thriller']
        Keywords: ['competition', 'secret', 'obsession', 'magic']
        Origin: en | Pop: 74.4
------------------------------
[0.532] The Dark Knight (2008)
        Director: Christopher Nolan
        Genre: ['Drama', 'Action', 'Crime']
        Keywords: ['dc comics', 'crime fighter', 'secret identity', 'scarecrow']
        Origin: en | Pop: 187.3
------------------------------
[0.522] Batman Begins (2005)
